# Day 069 — Exercise 2: Strip JSON from LLM Response

**What you'll build:** `strip_json_from_response(response) -> str` — extract the raw JSON string from any LLM response format.

**Why it matters:** Vision LLMs often wrap their JSON output in markdown code blocks (` ```json...``` `) or precede it with explanation text. This function is the noise filter that delivers a clean JSON string to `json.loads` regardless of the model's formatting habits.

In [ ]:
import re
import json


## Task

Implement `strip_json_from_response(response: str) -> str`:

1. Try `re.search(r'```(?:json)?\\s*([\\s\\S]*?)```', response)` — markdown code block
2. If found: return `block.group(1).strip()`
3. Otherwise: try `re.search(r'\\{[\\s\\S]*\\}', response)` — bare JSON
4. If found: return `obj.group(0).strip()`
5. If nothing found: `raise ValueError(f'No JSON found: {response[:200]!r}')`

## Your Implementation

In [ ]:
def strip_json_from_response(response: str) -> str:
    """Extract a JSON object from a raw LLM response string.

    Handles three formats:
        1. Markdown code block: ```json\n{...}\n```  (or ``` without 'json')
        2. Prefixed JSON:       'Here is the data: {...}'
        3. Plain JSON:          '{...}'

    Args:
        response: raw LLM output string
    Returns:
        Stripped JSON string ready for json.loads
    Raises:
        ValueError if no JSON object found in the response
    """
    raise NotImplementedError


In [ ]:
def strip_json_from_response(response: str) -> str:
    block = re.search(r'`{3}(?:json)?\s*([\s\S]*?)`{3}', response)
    if block:
        return block.group(1).strip()
    obj = re.search(r'\{[\s\S]*\}', response)
    if obj:
        return obj.group(0).strip()
    raise ValueError(f'No JSON found: {response[:200]!r}')


## Automated checks

In [ ]:
score, total = 0, 5
try:
    # Plain JSON
    r1 = strip_json_from_response('{"name": "Widget", "price": 9.99}')
    assert json.loads(r1) == {"name": "Widget", "price": 9.99}
    score += 1; print("\u2705 plain JSON")

    # Markdown code block with language tag
    r2 = strip_json_from_response('```json\n{"name": "Widget", "price": 9.99}\n```')
    assert json.loads(r2) == {"name": "Widget", "price": 9.99}
    score += 1; print("\u2705 markdown code block (```json)")

    # Markdown code block without language tag
    r3 = strip_json_from_response('```\n{"name": "Widget"}\n```')
    assert json.loads(r3) == {"name": "Widget"}
    score += 1; print("\u2705 markdown code block (no language tag)")

    # Prefixed with explanation text
    r4 = strip_json_from_response('Here is the data:\n{"name": "Widget", "price": 9.99}')
    assert json.loads(r4)['name'] == 'Widget'
    score += 1; print("\u2705 prefixed JSON (explanation text before)")

    # No JSON → ValueError
    raised = False
    try:
        strip_json_from_response("I cannot determine the values from this image.")
    except ValueError:
        raised = True
    assert raised, "Should raise ValueError when no JSON found"
    score += 1; print("\u2705 raises ValueError when no JSON object found")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def strip_json_from_response(response: str) -> str:
    block = re.search(r'`{3}(?:json)?\s*([\s\S]*?)`{3}', response)
    if block:
        return block.group(1).strip()
    obj = re.search(r'\{[\s\S]*\}', response)
    if obj:
        return obj.group(0).strip()
    raise ValueError(f'No JSON found: {response[:200]!r}')
```

**Why code blocks first?** If the response is ```` ```json\n{...}\n``` ````, the bare JSON regex `{[\\s\\S]*}` would also match — but it would include the backticks as surrounding text. Trying code blocks first and returning the capture group gives the clean inner content without backticks.

</details>